In [ ]:
import pandas as pd
import numpy as np
import re
import html
import os
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, ConfusionMatrixDisplay

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")

```markdown
## 1. Data Loading and Inspection
```

In [ ]:
# Load datasets (Update paths if running locally)
train_path = '/kaggle/input/datasets/riadhhossain/disaster-tweets/train (1).csv'
test_path = '/kaggle/input/datasets/riadhhossain/disaster-tweets/test (1).csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Training Set Shape: {train_df.shape}")
print(f"Test Set Shape: {test_df.shape}\n")

print("=== Target Distribution ===")
print(train_df['target'].value_counts(normalize=True).map('{:.1%}'.format))

```markdown
## 2. Exploratory Data Analysis (EDA)
```

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class Distribution
sns.countplot(x='target', data=train_df, ax=axes[0], palette='Set2')
axes[0].set_title('Class Distribution')
axes[0].set_xticklabels(['Not Disaster', 'Real Disaster'])

# Text Length Distribution
train_df['char_length'] = train_df['text'].str.len()
sns.histplot(data=train_df, x='char_length', hue='target', kde=True, ax=axes[1], palette='Set2')
axes[1].set_title('Text Length Distribution')

plt.tight_layout()
plt.show()

```markdown
## 3. Text Preprocessing
We apply conservative cleaning: removing URLs, mentions, and HTML entities, while preserving hashtags and punctuation for semantic value.
```

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = html.unescape(text)
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['cleaned_text'] = train_df['text'].apply(clean_text)
test_df['cleaned_text'] = test_df['text'].apply(clean_text)

print("Preprocessing complete.")
print("Example:", train_df['cleaned_text'].iloc[0])

```markdown
## 4. Train/Validation Split
```

In [ ]:
X = train_df['cleaned_text']
y = train_df['target']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")

```markdown
## 5. Feature Extraction (TF-IDF)
```

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)

print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

```markdown
## 6. Traditional Machine Learning Models
```

In [ ]:
models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    'Linear SVM': LinearSVC(random_state=RANDOM_STATE, max_iter=2000)
}

results = []

for name, model in models.items():
    start_time = time.time()
    model.fit(X_train_tfidf, y_train)
    train_time = time.time() - start_time
    
    y_pred = model.predict(X_val_tfidf)
    f1 = f1_score(y_val, y_pred)
    acc = accuracy_score(y_val, y_pred)
    
    results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'F1 Score': round(f1, 4),
        'Training Time (s)': round(train_time, 4)
    })
    print(f"{name} - F1: {f1:.4f} | Accuracy: {acc:.4f}")

results_df = pd.DataFrame(results)
display(results_df.style.highlight_max(subset=['F1 Score'], color='lightgreen'))

```markdown
## 7. Confusion Matrix & Feature Importance (Best Traditional Model)
```

In [ ]:
best_trad_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
best_trad_model.fit(X_train_tfidf, y_train)
y_pred_trad = best_trad_model.predict(X_val_tfidf)

# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(best_trad_model, X_val_tfidf, y_val, ax=ax, cmap='Blues')
plt.title('Confusion Matrix: Logistic Regression')
plt.show()

# Feature Importance
feature_names = tfidf_vectorizer.get_feature_names_out()
coefficients = best_trad_model.coef_[0]
feature_importance = pd.DataFrame({'feature': feature_names, 'coefficient': coefficients})

print("Top 10 Disaster Indicators:")
print(feature_importance.sort_values('coefficient', ascending=False).head(10))

```markdown
## 8. Advanced NLP: DistilBERT Fine-Tuning
```

In [ ]:
!pip install -q transformers datasets evaluate accelerate

import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

# 1. Prepare Data for Hugging Face (Must rename 'target' to 'labels')
bert_train_df, bert_val_df = train_test_split(
    train_df[['text', 'target']], test_size=0.2, random_state=42, stratify=train_df['target']
)
bert_train_df = bert_train_df.rename(columns={'target': 'labels'})
bert_val_df = bert_val_df.rename(columns={'target': 'labels'})

train_dataset = Dataset.from_pandas(bert_train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(bert_val_df.reset_index(drop=True))

# 2. Tokenization
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# 3. Load Model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 4. Define Metrics
f1_metric = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return f1_metric.compute(predictions=predictions, references=labels, average="binary")

# 5. Training Arguments
training_args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available()
)

# 6. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

print("Starting BERT fine-tuning...")
trainer.train()

In [ ]:
# Evaluate BERT
eval_results = trainer.evaluate()
print(f"DistilBERT Validation F1 Score: {eval_results['eval_f1']:.4f}")

# Generate Submission
test_dataset = Dataset.from_pandas(test_df[['id', 'text']].reset_index(drop=True))
tokenized_test = test_dataset.map(tokenize_function, batched=True)

predictions_output = trainer.predict(tokenized_test)
test_preds = np.argmax(predictions_output.predictions, axis=-1)

submission_df = pd.DataFrame({'id': test_df['id'], 'target': test_preds})
submission_df.to_csv('submission.csv', index=False)
print("Submission file saved.")
display(submission_df.head())

```markdown
## 9. Conclusion and Limitations

**Key Findings:**
1. Traditional models (Logistic Regression, Linear SVM) provide a strong, interpretable baseline with F1 scores around 0.77.
2. DistilBERT significantly improves performance (F1 > 0.80) by leveraging contextual embeddings to understand metaphorical language and sarcasm.

**Limitations:**
- TF-IDF ignores word order and deep semantic context.
- The dataset contains significant missing values in the `location` column (~33%).
- Social media language evolves rapidly; models may require frequent retraining.

**Future Work:**
- Implement larger Transformer models (RoBERTa, DeBERTa-v3).
- Utilize 5-Fold Cross-Validation for more robust evaluation.
- Incorporate multimodal data (text + images) if available.
```